In [1]:
import base64
import os
import io
from pdf2image import convert_from_path
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv("../.env")

True

In [3]:
# Configuration
API_KEY = os.getenv("OPENAI_API_KEY")
PDF_PATH = "../info-mascotas/Tenencia-Responsable.pdf"
OUTPUT_FILE = "Tenencia-Responsable.md"
MODEL = "gpt-5-mini" # Replace with "gpt-5-mini" if available

In [15]:
client = OpenAI(api_key=API_KEY)

def image_to_base64(image):
    buffered = io.BytesIO()
    image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

def clean_response(text):
    """Elimina los bloques de código ```markdown y maneja respuestas vacías"""
    # Eliminar encapsulamiento de código
    if text.startswith("```markdown"):
        text = text.replace("```markdown", "", 1)
    elif text.startswith("```"):
        text = text.replace("```", "", 1)
    
    if text.endswith("```"):
        text = text[:-3]

    return text.strip()

def transcribe_page(image, page_num):
    base64_image = image_to_base64(image)
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system", 
                "content": "You are a PDF to Markdown converter. Output ONLY the raw markdown text. Do not use code blocks. Do not converse."
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Transcribe this page exactly. Preserve headers, lists, and tables. If the page is blank, return an empty string."},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                ],
            }
        ]
    )
    return clean_response(response.choices[0].message.content)

def main():
    print(f"Processing {PDF_PATH}...")
    
    # Convert PDF to images (one per page)
    pages = convert_from_path(PDF_PATH)
    
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for i, page in enumerate(pages):
            print(f"Transcribing page {i+1}/{len(pages)}...")
            markdown_content = transcribe_page(page, i+1)
            
            # Write to file
            f.write(f"## Page {i+1}\n\n")
            f.write(markdown_content)
            f.write("\n\n---\n\n")
            
    print(f"Done! Saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Processing ../info-mascotas/Tenencia-Responsable.pdf...
Transcribing page 1/36...
Transcribing page 2/36...
Transcribing page 3/36...
Transcribing page 4/36...
Transcribing page 5/36...
Transcribing page 6/36...
Transcribing page 7/36...
Transcribing page 8/36...
Transcribing page 9/36...
Transcribing page 10/36...
Transcribing page 11/36...
Transcribing page 12/36...
Transcribing page 13/36...
Transcribing page 14/36...
Transcribing page 15/36...
Transcribing page 16/36...
Transcribing page 17/36...
Transcribing page 18/36...
Transcribing page 19/36...
Transcribing page 20/36...
Transcribing page 21/36...
Transcribing page 22/36...
Transcribing page 23/36...
Transcribing page 24/36...
Transcribing page 25/36...
Transcribing page 26/36...
Transcribing page 27/36...
Transcribing page 28/36...
Transcribing page 29/36...
Transcribing page 30/36...
Transcribing page 31/36...
Transcribing page 32/36...
Transcribing page 33/36...
Transcribing page 34/36...
Transcribing page 35/36...
Transcri